In [ ]:
# Set CUDA allocator config BEFORE importing torch so it actually takes effect.
# Reduces fragmentation, which matters when running SAM3 on a 16 GiB GPU.
# NOTE: requires kernel restart to take effect after first run.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, uuid, random, shutil, abc, contextlib, gc, copy, subprocess
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple, Union
from torchvision.transforms.functional import InterpolationMode

import uuid
from tqdm.auto import tqdm
import cv2
import numpy as np
import torch
import requests
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import torchvision.transforms.functional as TF
import itertools
from itertools import zip_longest
from dataclasses import replace
from PIL import Image
from transformers import pipeline

## 1. Data Structures

In [ ]:
@dataclass
class BoundingBox:
    xmin: int
    ymin: int
    xmax: int
    ymax: int

    @property
    def xyxy(self) -> List[float]:
        return [self.xmin, self.ymin, self.xmax, self.ymax]

    @property
    def width(self) -> int:
        return self.xmax - self.xmin

    @property
    def height(self) -> int:
        return self.ymax - self.ymin

    @property
    def area(self) -> int:
        return self.width * self.height

    @property
    def center(self) -> Tuple[int, int]:
        return (self.xmin + self.xmax) // 2, (self.ymin + self.ymax) // 2


@dataclass
class DetectionResult:
    score: float
    label: str
    box: BoundingBox
    mask: Optional[np.ndarray] = None

    @classmethod
    def from_dict(cls, d: Dict) -> "DetectionResult":
        return cls(
            score=d["score"],
            label=d["label"],
            box=BoundingBox(
                xmin=d["box"]["xmin"],
                ymin=d["box"]["ymin"],
                xmax=d["box"]["xmax"],
                ymax=d["box"]["ymax"],
            ),
        )

    @property
    def mask_area(self) -> int:
        if self.mask is None:
            return 0
        return int(np.asarray(self.mask).astype(bool).sum())

## 2. Config Dataclasses

In [ ]:
@dataclass
class DetectorConfig:
    """
    With SAM3, detection and segmentation are unified.
    Only `box_threshold` is used (as the SAM3 score threshold for filtering
    low-confidence detections).
    """
    model_id: str = "facebook/sam3"    # HF model id
    box_threshold: float = 0.25         # SAM3 score threshold

@dataclass
class SegmenterConfig:
    """
    SAM3 from HuggingFace Transformers (facebook/sam3).

    image_size: 1008 is the model-trained resolution (best accuracy).
                Lower values (560, 336) reduce memory at the cost of accuracy.
                For T4 / 16 GiB GPUs running tight on memory, try 560 first.
    mask_threshold: Threshold applied to predicted mask logits before
                    binarising (HF default is 0.5).
    """
    model_id: str = "facebook/sam3"
    image_size: int = 1008
    mask_threshold: float = 0.5
    polygon_refinement: bool = False


@dataclass
class CropConfig:
    """
    policy options
    --------------
    sliding_window  Find the square that covers the most mask pixels (integral-image trick).
    bbox            Crop directly from the detection bounding box, squared up.
    center_mask     Crop a square centred on the mask centroid.
    largest_bbox    Pick the detection with the largest bbox, ignore masks.

    size_bias       Controls random size: side = min_win + u^bias * (req_win - min_win).
                    bias=1 uniform; bias>1 skews toward min_win; bias<1 toward req_win.

    mask_selection  Which detection to use: largest_area | highest_score | first
    """
    policy: str = "sliding_window"
    min_win: int = 64
    req_win: int = 224
    size_bias: float = 1.5
    mask_selection: str = "highest_score"


@dataclass
class AugmentationConfig:
    enabled: bool = True
    rotation: bool = True
    shearing: bool = True
    flipping: bool = True
    brightness: bool = True
    contrast: bool = True
    saturation: bool = True
    hue: bool = True
    brightness_range: Tuple[float, float] = (0.6, 1.4)
    contrast_range: Tuple[float, float] = (0.5, 1.5)
    saturation_range: Tuple[float, float] = (0.5, 1.5)
    hue_range: Tuple[float, float] = (-0.3, 0.3)
    shear_range: Tuple[float, float] = (-15, 15)

@dataclass
class DatasetConfig:
    base_path: str = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train"
    images_per_class: int = 10000      # max source images to try
    target_images_per_concept: int = 120   # desired final count
    extensions: Tuple[str, ...] = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")

@dataclass
class PipelineConfig:
    detector: DetectorConfig = field(default_factory=DetectorConfig)
    segmenter: SegmenterConfig = field(default_factory=SegmenterConfig)
    crop: CropConfig = field(default_factory=CropConfig)
    augmentation: AugmentationConfig = field(default_factory=AugmentationConfig)
    dataset: DatasetConfig = field(default_factory=DatasetConfig)
    device: str = "auto"   # "auto" -> cuda if available, else cpu
    crops_path: str = "/kaggle/working/crops"
    augmented_path: str = "/kaggle/working/augmented_data"

## 3. Low-Level Utils

In [ ]:
def load_image(image_str: str) -> Image.Image:
    if image_str.startswith("http"):
        return Image.open(requests.get(image_str, stream=True).raw).convert("RGB")
    return Image.open(image_str).convert("RGB")


def get_boxes(results: List[DetectionResult]) -> List[List[List[float]]]:
    """Boxes in the nested format expected by the SAM processor."""
    return [[r.box.xyxy for r in results]]


def mask_to_polygon(mask: np.ndarray) -> List[List[int]]:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return max(contours, key=cv2.contourArea).reshape(-1, 2).tolist()


def polygon_to_mask(polygon: List[Tuple[int, int]], image_shape: Tuple[int, int]) -> np.ndarray:
    mask = np.zeros(image_shape, dtype=np.uint8)
    cv2.fillPoly(mask, [np.array(polygon, dtype=np.int32)], color=(255,))
    return mask


def refine_masks(masks: torch.BoolTensor, polygon_refinement: bool = False) -> List[np.ndarray]:
    masks = masks.cpu().float().permute(0, 2, 3, 1).mean(axis=-1)
    masks = (masks > 0).int().numpy().astype(np.uint8)
    masks = list(masks)
    if polygon_refinement:
        for i, mask in enumerate(masks):
            masks[i] = polygon_to_mask(mask_to_polygon(mask), mask.shape)
    return masks


def resolve_device(device: str) -> str:
    return ("cuda" if torch.cuda.is_available() else "cpu") if device == "auto" else device


def _resolve_output_path(out_path: str) -> str:
    if out_path.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff")):
        os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
        return out_path
    os.makedirs(out_path, exist_ok=True)
    return os.path.join(out_path, f"crop_{uuid.uuid4().hex[:8]}.png")


def _pick_best_detection(
    detections: List[DetectionResult], strategy: str
) -> Optional[DetectionResult]:
    valid = [d for d in detections if d.mask is not None and d.mask_area > 0]
    if not valid:
        return None
    if strategy == "highest_score":
        return max(valid, key=lambda d: d.score)
    if strategy == "first":
        return valid[0]
    return max(valid, key=lambda d: d.mask_area)  # "highest_score" (default)

## 4. Crop Policies

Available: `sliding_window`, `bbox`, `center_mask`, `largest_bbox`

All policies share the same interface — swap `CropConfig.policy` with no other changes.

In [ ]:
class CropPolicyBase(abc.ABC):
    @abc.abstractmethod
    def crop(self, image: np.ndarray, detection: DetectionResult, config: CropConfig) -> Tuple[int, int, int]:
        """Return (x1, y1, side)."""

    def execute(
        self,
        image: np.ndarray,
        detections: List[DetectionResult],
        config: CropConfig,
        out_path: str,
    ) -> Tuple[str, Tuple[int, int, int]]:
        best = _pick_best_detection(detections, config.mask_selection)
        if best is None:
            raise ValueError("No valid detections with non-empty masks.")
        x1, y1, side = self.crop(image, best, config)
        img_pil = Image.fromarray(image)
        crop = img_pil.crop((x1, y1, x1 + side, y1 + side))
        if side != config.req_win:
            crop = crop.resize((config.req_win, config.req_win), resample=Image.BICUBIC)
        fn = _resolve_output_path(out_path)
        crop.save(fn)
        return fn, (x1, y1, side)


class SlidingWindowCropPolicy(CropPolicyBase):
    """Integral-image sliding window that maximises mask coverage."""

    @staticmethod
    def _get_window(mask: np.ndarray, win: int) -> Tuple[int, int, int]:
        m = np.asarray(mask).astype(np.uint8)
        H, W = m.shape
        win = min(win, H, W)
        ii = np.pad(m, ((1, 0), (1, 0)), mode="constant").cumsum(0).cumsum(1)
        sums = ii[win:, win:] - ii[:-win, win:] - ii[win:, :-win] + ii[:-win, :-win]
        maxv = int(sums.max())
        ys, xs = np.where(sums == maxv)
        k = np.random.randint(len(xs))
        return int(xs[k]), int(ys[k]), maxv

    def crop(self, image, detection, config):
        H, W = image.shape[:2]
        upper = min(config.req_win, H, W)
        lower = min(config.min_win, upper)
        t = random.random() ** config.size_bias
        side = int(round(lower + t * (upper - lower)))
        x1, y1, _ = self._get_window(detection.mask, side)
        return x1, y1, side


class BBoxCropPolicy(CropPolicyBase):
    """Crop the detection's bounding box, squared up and clamped."""

    def crop(self, image, detection, config):
        H, W = image.shape[:2]
        box = detection.box
        cx, cy = box.center
        side = max(min(max(box.width, box.height), H, W), config.min_win)
        x1 = max(0, min(cx - side // 2, W - side))
        y1 = max(0, min(cy - side // 2, H - side))
        return x1, y1, side


class CenterMaskCropPolicy(CropPolicyBase):
    """Crop a square centred on the mask centroid (falls back to bbox centre)."""

    def crop(self, image, detection, config):
        H, W = image.shape[:2]
        side = min(config.req_win, H, W)
        if detection.mask is not None and detection.mask_area > 0:
            ys, xs = np.where(np.asarray(detection.mask).astype(bool))
            cx, cy = int(xs.mean()), int(ys.mean())
        else:
            cx, cy = detection.box.center
        x1 = max(0, min(cx - side // 2, W - side))
        y1 = max(0, min(cy - side // 2, H - side))
        return x1, y1, side


class LargestBBoxCropPolicy(CropPolicyBase):
    """Pick detection with the largest bbox area; masks not required."""

    def execute(self, image, detections, config, out_path):
        if not detections:
            raise ValueError("No detections provided.")
        best = max(detections, key=lambda d: d.box.area)
        x1, y1, side = self.crop(image, best, config)
        img_pil = Image.fromarray(image)
        crop = img_pil.crop((x1, y1, x1 + side, y1 + side))
        if side != config.req_win:
            crop = crop.resize((config.req_win, config.req_win), resample=Image.BICUBIC)
        fn = _resolve_output_path(out_path)
        crop.save(fn)
        return fn, (x1, y1, side)

    def crop(self, image, detection, config):
        H, W = image.shape[:2]
        cx, cy = detection.box.center
        side = max(min(max(detection.box.width, detection.box.height), H, W), config.min_win)
        x1 = max(0, min(cx - side // 2, W - side))
        y1 = max(0, min(cy - side // 2, H - side))
        return x1, y1, side


_CROP_POLICY_REGISTRY: Dict[str, type] = {
    "sliding_window": SlidingWindowCropPolicy,
    "bbox": BBoxCropPolicy,
    "center_mask": CenterMaskCropPolicy,
    "largest_bbox": LargestBBoxCropPolicy,
}


class Cropper:
    """Delegates cropping to the policy named in CropConfig.policy."""

    def __init__(self, config: CropConfig):
        self.config = config
        cls = _CROP_POLICY_REGISTRY.get(config.policy)
        if cls is None:
            raise ValueError(f"Unknown crop policy '{config.policy}'. Available: {list(_CROP_POLICY_REGISTRY)}")
        self._policy: CropPolicyBase = cls()

    def crop(self, image: np.ndarray, detections: List[DetectionResult], out_path: str):
        return self._policy.execute(image, detections, self.config, out_path)

## 5. Augmenter

In [ ]:
class Augmenter:
    def __init__(self, config):
        self.cfg = config

    def augment_one(self, img: Image.Image) -> Image.Image:
        original_size = img.size  # (width, height)
        # FIX 3 — snapshot per il safety-check finale
        original_array = np.array(img)

        # ------------------------
        # 1. ROTATION (restricted)
        # ------------------------
        if self.cfg.rotation:
            angle = random.choice([90, 180, -90, 0])
            img = TF.rotate(img, angle)

        # ------------------------
        # 2. SHEARING
        # ------------------------
        if self.cfg.shearing:
            shear_x = random.uniform(*self.cfg.shear_range)
            img = TF.affine(
                img,
                angle=0.0,
                translate=[0, 0],
                scale=1.0,
                shear=[shear_x, 0.0],
                interpolation=InterpolationMode.BILINEAR
            )

        # ------------------------
        # 3. CROP + RESIZE
        # ------------------------
        width, height = img.size
        crop_h = int(height * 0.75)
        crop_w = int(width * 0.75)

        img = TF.center_crop(img, (crop_h, crop_w))
        img = TF.resize(img, (original_size[1], original_size[0]))

        # ------------------------
        # 4. FLIPPING
        # ------------------------
        if self.cfg.flipping:
            flip = random.choice(["h", "v", "hv", "none"])
            if flip in ("h", "hv"):
                img = TF.hflip(img)
            if flip in ("v", "hv"):
                img = TF.vflip(img)

        # ------------------------
        # 5. COLOR AUGMENTATION
        # ------------------------

        if self.cfg.brightness:
            factor = random.uniform(*self.cfg.brightness_range)
            img = TF.adjust_brightness(img, factor)

        if self.cfg.contrast:
            factor = random.uniform(*self.cfg.contrast_range)
            img = TF.adjust_contrast(img, factor)

        if self.cfg.saturation:
            factor = random.uniform(*self.cfg.saturation_range)
            img = TF.adjust_saturation(img, factor)

        if self.cfg.hue:
            # NOTE: hue expects range roughly [-0.5, 0.5]
            delta = random.uniform(*self.cfg.hue_range)
            img = TF.adjust_hue(img, delta)

        if np.array_equal(original_array, np.array(img)):
            img = TF.hflip(img)

        return img

## 6. Extractor

In [ ]:
# Install SAM3 from facebookresearch/sam3 (the official Meta repo).
# We use the source install because the HuggingFace Transformers wrapper for
# SAM3 in late 2025 / early 2026 (3.1 checkpoint) has an unresolved checkpoint key-mapping bug
# that ships the text encoder with random weights — every text prompt returns
# garbage scores. The source path uses the original checkpoint format directly
# and avoids that issue entirely.
#
# Pre-flight checklist:
#  - Accept the license at https://huggingface.co/facebook/sam3
#  - Add an HF token to Kaggle Secrets under the key "HF_TOKEN"
#
# This cell is idempotent: safe to re-run, will skip steps already done.

SAM3_DIR = "/kaggle/working/sam3"

if not os.path.isdir(SAM3_DIR):
    print(f"Cloning sam3 to {SAM3_DIR}...")
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/facebookresearch/sam3.git", SAM3_DIR],
        check=True,
    )
else:
    print(f"sam3 already cloned at {SAM3_DIR}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
     "--ignore-requires-python", "-e", SAM3_DIR],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "ftfy", "iopath>=0.1.10", "decord", "ipycanvas"],
    check=True,
)

if SAM3_DIR not in sys.path:
    sys.path.insert(0, SAM3_DIR)

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
import sam3 as _sam3_pkg
print(f"sam3 imported from: {_sam3_pkg.__file__}")

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")

In [ ]:
class SAM3Model:
    """
    SAM3-based Promptable Concept Segmentation using the official
    facebookresearch/sam3 source distribution.
    
    Memory strategy on T4 (16 GiB):
        1. Cast the model to float16 with `.half()` after build. Weights drop
           from ~3.4 GiB (fp32) to ~1.7 GiB (fp16); activations of the ViT-L at
           1008x1008 also halve.
        2. Wrap inference in `torch.amp.autocast(bfloat16)` so any internal
           ops that need a wider dtype still work.
        3. Call `set_image` once per text prompt — required because the SAM3
           processor mutates `state["backbone_out"]` and `set_text_prompt`
           refuses to run a second time on the same state. We tested keeping
           state across prompts; the source code explicitly disallows it.
        4. Free CUDA cache after every label and every image to avoid the
           per-iteration ViT activations accumulating.

    The class keeps the original detect / segment / process interface so the
    rest of the pipeline (Cropper, Pipeline.run, etc.) works unchanged.
    """

    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.device = resolve_device(config.device)

        # build_sam3_image_model downloads the checkpoint from HuggingFace
        # (facebook/sam3) on first call and caches it. Requires HF login.
        sam3_model = build_sam3_image_model(device=self.device)

        # Convert to float16 on CUDA: ~halves both weights and activations.
        # On CPU we stay at fp32 because fp16 CPU ops are slow.
        if self.device == "cuda":
            sam3_model = sam3_model
            self.dtype = torch.float16
        else:
            self.dtype = torch.float32

        sam3_model.eval()
        self.processor = Sam3Processor(sam3_model)

    def _autocast_ctx(self):
        """Use autocast to handle any internal ops that need mixed precision."""
        if self.device == "cuda":
            return torch.amp.autocast(device_type="cuda", dtype=self.dtype)
        return contextlib.nullcontext()

    def _free_gpu(self) -> None:
        if self.device == "cuda":
            gc.collect()
            torch.cuda.empty_cache()

    @staticmethod
    def _normalise_label(label: str) -> str:
        return label.rstrip(".").strip()

    @staticmethod
    def _to_2d_mask(mask: np.ndarray, target_hw: Tuple[int, int]) -> np.ndarray:
        m = np.asarray(mask)
        while m.ndim > 2:
            m = m.squeeze(0)
        m = (m > 0).astype(np.uint8)
        H, W = target_hw
        if m.shape != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
        return m

    def _run_sam3(
        self, image: Image.Image, label: str
    ) -> Optional[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
        """
        Run one SAM3 forward pass. Returns (boxes, masks, scores) as numpy.
        Returns None on error. Always frees GPU memory before returning.
        """
        state = None
        result = None
        try:
            with torch.inference_mode(), self._autocast_ctx():
                state = self.processor.set_image(image)
                result = self.processor.set_text_prompt(state=state, prompt=label)
                boxes  = result["boxes"].detach().float().cpu().numpy()
                masks  = result["masks"].detach().float().cpu().numpy()
                scores = result["scores"].detach().float().cpu().numpy()
            return boxes, masks, scores
        except torch.cuda.OutOfMemoryError as e:
            print(f"  [SAM3 OOM] '{label}': {e}")
            return None
        except Exception as e:
            print(f"  [SAM3 prompt error] '{label}': {type(e).__name__}: {e}")
            return None
        finally:
            del state, result
            self._free_gpu()

    def detect(
        self,
        image: Image.Image,
        labels: List[str],
        threshold: Optional[float] = None,
    ) -> List[DetectionResult]:
        threshold = threshold if threshold is not None else self.cfg.detector.box_threshold
        img_W, img_H = image.size
        detections: List[DetectionResult] = []

        for raw_label in labels:
            label = self._normalise_label(raw_label)
            out = self._run_sam3(image, label)
            if out is None:
                continue
            boxes, masks, scores = out

            for box, mask, score in zip(boxes, masks, scores):
                if float(score) < threshold:
                    continue
                x1 = max(0, min(int(round(float(box[0]))), img_W - 1))
                y1 = max(0, min(int(round(float(box[1]))), img_H - 1))
                x2 = max(0, min(int(round(float(box[2]))), img_W))
                y2 = max(0, min(int(round(float(box[3]))), img_H))
                if x2 <= x1 or y2 <= y1:
                    continue

                m_2d = self._to_2d_mask(mask, (img_H, img_W))
                if self.cfg.segmenter.polygon_refinement and m_2d.sum() > 0:
                    m_2d = polygon_to_mask(mask_to_polygon(m_2d), m_2d.shape)

                detections.append(DetectionResult(
                    score=float(score),
                    label=label,
                    box=BoundingBox(xmin=x1, ymin=y1, xmax=x2, ymax=y2),
                    mask=m_2d,
                ))

        return detections

    def segment(
        self,
        image: Image.Image,
        detections: List[DetectionResult],
    ) -> List[DetectionResult]:
        """No-op for SAM3: detect() already returns masks."""
        return detections

    def process(
        self,
        image: Union[Image.Image, str],
        labels: List[str],
        threshold: Optional[float] = None,
    ) -> Tuple[np.ndarray, List[DetectionResult]]:
        if isinstance(image, str):
            image = load_image(image)
        elif not isinstance(image, Image.Image):
            raise TypeError(f"Expected PIL.Image or str, got {type(image)}")
        detections = self.detect(image, labels, threshold)
        detections = self.segment(image, detections)
        return np.array(image), detections

## 7. Dataset Collector

In [ ]:
class DatasetCollector:
    """
    Collects image paths from an ImageNet-style tree.
    class_mapping: {concept: [synset_ids]}

    Quando un concetto è mappato a più synset (es. case "correlated" o
    "random"), le immagini vengono prese **da tutte le classi** in modo
    bilanciato:
      - quota per-synset equa (budget // N, con resto sui primi synset);
      - redistribuzione del deficit: se un synset ha meno immagini della
        sua quota, le mancanti vengono prese dai synset con surplus;
      - interleave round-robin sull\'output, così che il loop di
        estrazione tocchi tutte le classi anche se non scorre l\'intera
        lista.
    """

    def __init__(self, config: DatasetConfig, class_mapping: Dict[str, List[str]]):
        self.cfg = config
        self.class_mapping = class_mapping

    def collect(self) -> Dict[str, List[str]]:
        ext_lower = tuple(e.lower() for e in self.cfg.extensions)
        result: Dict[str, List[str]] = {}

        for concept, synsets in self.class_mapping.items():
            if not synsets:
                result[concept] = []
                print(f"[DatasetCollector] {concept}: 0 images (no synsets).")
                continue

            # 1) Raccogli tutte le immagini disponibili PER synset
            per_synset: List[List[str]] = []
            for synset in synsets:
                input_dir = os.path.join(self.cfg.base_path, synset)
                if not os.path.isdir(input_dir):
                    print(f"[DatasetCollector] Warning: {input_dir} not found.")
                    per_synset.append([])
                    continue
                
                found: List[str] = []
                for dp, _, fns in os.walk(input_dir):
                    # Rimuoviamo il sorted() e raccogliamo i path
                    for fn in fns: 
                        if fn.lower().endswith(ext_lower):
                            found.append(os.path.join(dp, fn))
                
                # --- LA MODIFICA CHIAVE ---
                # Mescoliamo le immagini all'interno del singolo synset
                random.shuffle(found) 
                per_synset.append(found)

            # 2) Quota equa per synset
            budget = self.cfg.images_per_class
            n = len(synsets)
            base_q, extra = divmod(budget, n)
            quotas = [base_q + (1 if i < extra else 0) for i in range(n)]

            taken: List[List[str]] = [paths[:q] for paths, q in zip(per_synset, quotas)]
            surplus: List[List[str]] = [paths[q:] for paths, q in zip(per_synset, quotas)]

            # 3) Redistribuisci il deficit ai synset con surplus (round-robin)
            deficit = budget - sum(len(t) for t in taken)
            i = 0
            safety = budget * n + 1
            while deficit > 0 and any(surplus) and safety > 0:
                if surplus[i % n]:
                    taken[i % n].append(surplus[i % n].pop(0))
                    deficit -= 1
                i += 1
                safety -= 1

            # 4) Interleave round-robin per garantire varietà nell'ordine di processing
            interleaved: List[str] = []
            for tup in zip_longest(*taken, fillvalue=None):
                for p in tup:
                    if p is not None:
                        interleaved.append(p)

            result[concept] = interleaved
            breakdown = ", ".join(f"{s}={len(t)}" for s, t in zip(synsets, taken))
            print(f"[DatasetCollector] {concept}: {len(interleaved)} images ({breakdown})")

        return result

## 8. Main Pipeline

In [ ]:
class Pipeline:
    """
    End-to-end pipeline: load -> detect (SAM3) -> crop -> augment.

    The expensive SAM3 model is loaded once in __init__ and reused across all
    experiments. Crop policy and augmentation toggle can be overridden per-call
    via `run_experiment()`, so multiple datasets (ablation studies) can be
    produced from one pipeline instance without reloading the model.

    Usage
    -----
        pipe = Pipeline(config)

        # Vanilla run with config defaults:
        pipe.run_experiment("vanilla", CLASS_MAPPING_BASE, LABELS_BASE)

        # Override crop policy:
        pipe.run_experiment("crop_bbox", CLASS_MAPPING_BASE, LABELS_BASE,
                            crop_policy="bbox")

        # Turn augmentation on/off:
        pipe.run_experiment("no_aug", ..., augmentation_enabled=False)
    """

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.model = SAM3Model(config)
        # No pre-built cropper/augmenter; rebuilt per experiment if overridden.
        self._default_cropper = Cropper(config.crop)
        self._default_augmenter = Augmenter(config.augmentation)

    # ---------- helpers ----------

    def _reset_output_dirs(self, *dirs: str) -> None:
        for d in dirs:
            if os.path.exists(d):
                shutil.rmtree(d)
            os.makedirs(d, exist_ok=True)

    def _build_cropper(self, crop_policy: Optional[str]) -> "Cropper":
        if crop_policy is None or crop_policy == self.config.crop.policy:
            return self._default_cropper
        crop_cfg = replace(self.config.crop, policy=crop_policy)
        return Cropper(crop_cfg)

    def _build_augmenter(self, augmentation_enabled: Optional[bool]) -> "Augmenter":
        if augmentation_enabled is None or augmentation_enabled == self.config.augmentation.enabled:
            return self._default_augmenter
        aug_cfg = replace(self.config.augmentation, enabled=augmentation_enabled)
        return Augmenter(aug_cfg)

    def _process_image(
        self,
        image_path: str,
        labels: List[str],
        out_dir: str,
        cropper: "Cropper",
        threshold: Optional[float] = None,
    ) -> Optional[Tuple[str, Tuple[int, int, int]]]:
        try:
            image_array, detections = self.model.process(image_path, labels, threshold)
        except Exception as e:
            print(f"  [model error] {image_path}: {e}")
            return None
        if not detections:
            return None
        try:
            return cropper.crop(image_array, detections, out_dir)
        except ValueError as e:
            print(f"  [crop error] {image_path}: {e}")
            return None

    # ---------- main entry point ----------

    

    # ---------- backward-compat wrapper ----------

    def run(self, class_mapping, labels, crop_type, threshold=None):
        """Backward-compat alias for run_experiment(name=crop_type, ...)."""
        return self.run_experiment(
            name=crop_type,
            class_mapping=class_mapping,
            labels=labels,
            threshold=threshold,
        )

    def process_image(self, image_path, labels, out_dir, threshold=None):
        """Backward-compat alias — uses the default cropper."""
        return self._process_image(image_path, labels, out_dir, self._default_cropper, threshold)

## 9. Visualisation Utils

In [ ]:
def annotate(image: Union[Image.Image, np.ndarray], detections: List[DetectionResult]) -> np.ndarray:
    img = np.array(image) if isinstance(image, Image.Image) else image
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    for det in detections:
        color = np.random.randint(0, 256, size=3).tolist()
        b = det.box
        cv2.rectangle(img, (b.xmin, b.ymin), (b.xmax, b.ymax), color, 2)
        cv2.putText(img, f"{det.label}: {det.score:.2f}", (b.xmin, b.ymin - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        if det.mask is not None:
            contours, _ = cv2.findContours((det.mask * 255).astype(np.uint8),
                                           cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(img, contours, -1, color, 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def plot_detections(
    image: Union[Image.Image, np.ndarray],
    detections: List[DetectionResult],
    save_name: Optional[str] = None,
) -> None:
    plt.imshow(annotate(image, detections))
    plt.axis("off")
    if save_name:
        plt.savefig(save_name, bbox_inches="tight")
    plt.show()


def plot_images(rows: int, cols: int, directory: str) -> None:
    """Fixed: uses os.path.join instead of string concatenation."""
    exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
    files = [f for f in os.listdir(directory) if f.lower().endswith(exts)]
    if not files:
        print(f"No images in {directory}")
        return
    fig = plt.figure(figsize=(cols * 3, rows * 3))
    for i in range(rows * cols):
        ax = fig.add_subplot(rows, cols, i + 1)
        ax.imshow(Image.open(os.path.join(directory, random.choice(files))))
        ax.axis("off")
        ax.set_title(f"Image {i+1}", fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_detections_plotly(
    image: np.ndarray,
    detections: List[DetectionResult],
    class_colors: Optional[Dict[int, str]] = None,
) -> None:
    named_colors = [
        "aqua","blue","blueviolet","brown","cadetblue","chartreuse","chocolate",
        "coral","cornflowerblue","crimson","cyan","darkblue","darkcyan","darkgoldenrod",
        "darkgreen","darkorange","darkorchid","darkred","darkseagreen","darkturquoise",
        "darkviolet","deeppink","deepskyblue","dodgerblue","firebrick","forestgreen",
        "fuchsia","gold","goldenrod","green","hotpink","indianred","indigo","lawngreen",
        "limegreen","magenta","maroon","mediumblue","mediumorchid","mediumpurple",
        "mediumseagreen","mediumturquoise","mediumvioletred","midnightblue","navy",
        "olive","orange","orangered","orchid","peru","pink","plum","purple","red",
        "rosybrown","royalblue","saddlebrown","salmon","seagreen","sienna","skyblue",
        "slateblue","springgreen","steelblue","tan","teal","tomato","turquoise",
        "violet","yellowgreen",
    ]
    if class_colors is None:
        sample = random.sample(named_colors, min(len(detections), len(named_colors)))
        class_colors = {i: c for i, c in enumerate(sample)}

    fig = px.imshow(image)
    shapes = []
    for idx, det in enumerate(detections):
        color = class_colors.get(idx, "red")
        if det.mask is not None:
            polygon = mask_to_polygon(det.mask)
            fig.add_trace(go.Scatter(
                x=[p[0] for p in polygon] + [polygon[0][0]],
                y=[p[1] for p in polygon] + [polygon[0][1]],
                mode="lines", line=dict(color=color, width=2),
                fill="toself", name=f"{det.label}: {det.score:.2f}",
            ))
        xmin, ymin, xmax, ymax = det.box.xyxy
        shapes.append(dict(type="rect", xref="x", yref="y",
                           x0=xmin, y0=ymin, x1=xmax, y1=ymax,
                           line=dict(color=color)))

    buttons = (
        [dict(label="None", method="relayout", args=["shapes", []])]
        + [dict(label=f"Det {i+1}", method="relayout", args=["shapes", [s]]) for i, s in enumerate(shapes)]
        + [dict(label="All", method="relayout", args=["shapes", shapes])]
    )
    fig.update_layout(
        xaxis=dict(visible=False), yaxis=dict(visible=False), showlegend=True,
        updatemenus=[dict(type="buttons", direction="up", buttons=buttons)],
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()

## 10. Example Usage

In [ ]:
config = PipelineConfig(
    # FIX 1 — threshold abbassato da 0.4 a 0.25.
    # 0.4 era troppo alto per prompt di texture astratta (es. "striped pattern"):
    # SAM3 assegna spesso score 0.25-0.39 a texture che non sono oggetti discreti,
    # producendo pochissimi crop. 0.25 è il floor consigliato per questo dominio.
    detector=DetectorConfig(box_threshold=0.25),
    segmenter=SegmenterConfig(polygon_refinement=False),
    crop=CropConfig(
        policy="sliding_window",   # default; overridden per-experiment below
        min_win=64,
        req_win=224,
        size_bias=1.5,
        mask_selection="highest_score",
    ),
    augmentation=AugmentationConfig(
        enabled=True,
        rotation=True, shearing=True, flipping=True,
        brightness=True, contrast=True, saturation=True, hue=True,
        brightness_range=(0.6, 1.4),
        contrast_range=(0.5, 1.5),
        saturation_range=(0.5, 1.5),
        hue_range=(-0.3, 0.3),
        shear_range=(-15, 15),
    ),
    dataset=DatasetConfig(
        base_path="/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train",
        images_per_class=10000,
        target_images_per_concept=120,
    ),
    device="auto",
    crops_path="/kaggle/working/crops",
    augmented_path="/kaggle/working/augmented_data",
)

In [ ]:
pipe = Pipeline(config)

In [ ]:
SYNSET_MAPPING = {
    "striped": {
        "single": ["n02391049"], # zebra
        "multi": ["n02391049", "n02129604"], # zebra, tiger
        "random": ["n02134084", "n03942813", "n03584254", "n03481172"] # ice bear, ping-pong ball, iPod, hammer
    },
    "dotted": {
        "single": ["n02110341"], # dalmatian
        "multi": ["n02110341", "n02165456"], # dalmatian, ladybug
        "random": ["n02445715", "n03930313", "n03271574", "n04536866"] # skunk, picket fence, electric fan, violin
    },
    "chequered": {
        "single": ["n06785654"], # crossword puzzle
        "multi": ["n06785654", "n04033995"], # crossword puzzle, quilt
        "random": ["n09468604", "n01910747", "n04069434", "n03792782"] # valley, jellyfish, reflex camera, mountain bike
    },
    "wood": {
        "single": ["n03930313"], # picket fence
        "multi": ["n04597913", "n03891251"], # wooden spoon, park bench
        "random": ["n04311004", "n04589890", "n01910747", "n09472597"] # steel arch bridge, window screen, jellyfish, volcano
    },
    "water": {
        "single": ["n09332890", "n03388043"], # lakeside, fountain
        "multi": ["n09421951", "n09332890", "n03388043"], # sandbar, lakeside, fountain
        "random": ["n03347037", "n03661043", "n04154565"] # fire screen, library, screwdriver
    },
    "braided": {
        "single": ["n03482405"], # hamper
        "multi": ["n03482405", "n03627232", "n07695742"], # knot, pretzel, hamper
        "random": ["n03982430", "n02088466", "n07715103"] # billiard table, bloodhound, cauliflower
    },
    "bubbly": {
        "single": ["n09229709"], # bubble
        "multi": ["n09229709", "n02823750", "n09428293"], # bubble, beer glass, seashore
        "random": ["n04326547", "n04482393", "n02280649"] # stone wall, tricycle, cabbage butterfly
    },
    "fibrous": {
        "single": ["n07802026"],  # hay
        "multi": ["n07802026", "n04584207", "n02906734"], # hay, wig, broom
        "random": ["n04254680", "n03196217", "n06874185", "n03345487", "n09472597"] # soccer ball, digital clock, traffic light, fire engine, volcano"
    },
    "veined": {
        "single": ["n07714571"],  # cabbage (head_cabbage)
        "multi": ["n07714571", "n01917289"], # cabbage, brain coral
        "random": ["n02802426", "n04146614", "n03063599", "n04442312", "n01910747"] # basketball, school bus, coffee mug, toaster, jellyfish\n"
    }
}

LABELS_BASE = {
    "striped":   ["striped pattern", "stripes", "striped texture"],
    "dotted":    ["dotted pattern", "dots", "dotted texture", "spotted pattern", "spots", "polka dots", "spotted texture"],
    "chequered": ["checkered pattern","checkerboard", "chequered pattern"],
    "wood":      ["wooden material", "wood", "wooden texture", "wooden object"],
    "water":     ["water element", "water", "water texture"],
    "braided":   ["braided pattern", "braided", "braided texture", "braided object"],
    "bubbly":    ["bubbly pattern", "bubbles", "bubbly texture", "bubbly object"],
    "fibrous":   ["fibrous pattern", "fibers", "fibrous texture", "fibrous object"],
    "veined":    ["veined pattern", "veins", "veined texture", "veined object", "marble texture", "leaf veins", "veined marble",
                    "organic vein pattern"]
}

Prompt Variations below:

In [ ]:
PROMPT_VARIANTS = {
    # Type 1: minimal bare keyword (highest recall, least constrained)
    "bare_term": {
        "striped":   ["stripes"],
        "dotted":    ["dots"],
        "chequered": ["checkerboard"],
        "wood":      ["wood"],
        "water":     ["water"],
        "braided":   ["braid"],
        "bubbly":    ["bubbles"],
        "fibrous":   ["fibers"],
        "veined":    ["veins"],
    },
    # Type 2: natural-language surface phrase (the original "vanilla" wording)
    "surface_phrase": {
        "striped":   ["a striped surface"],
        "dotted":    ["a dotted surface"],
        "chequered": ["a chequered surface"],
        "wood":      ["a wooden surface"],
        "water":     ["a body of water"],
        "braided":   ["a braided surface"],
        "bubbly":    ["a bubbly surface"],
        "fibrous":   ["a fibrous surface"],
        "veined":    ["a veined surface"],
    },
    # Type 3: explicit texture phrasing (emphasises the texture concept)
    "texture_phrase": {
        "striped":   ["striped texture"],
        "dotted":    ["dotted texture"],
        "chequered": ["chequered texture"],
        "wood":      ["wooden texture"],
        "water":     ["water texture"],
        "braided":   ["braided texture"],
        "bubbly":    ["bubbly texture"],
        "fibrous":   ["fibrous texture"],
        "veined":    ["veined texture"],
    },
}

## 11. Ablation Experiments

### 11. Full Experiment

In [ ]:
def generate_dual_dataset(pipe: Pipeline, target_imgs: int = 120, mode: str = "vanilla"):
    POLICIES = ["sliding_window", "bbox", "center_mask", "largest_bbox"]
    TARGET_CONCEPTS = ["striped", "dotted", "chequered", "wood", "water", "braided", "bubbly", "fibrous", "veined"]
    
    # 1. Barra di progresso principale sui concetti
    pbar_concepts = tqdm(TARGET_CONCEPTS, desc=f"Pipeline ({mode})")
    
    for concept in pbar_concepts:
        # Aggiorna il testo sulla barra per mostrare cosa sta processando
        pbar_concepts.set_postfix(current_concept=concept)
        
        # Recupera i casi (single, multi, random) dal tuo dizionario unificato
        cases = SYNSET_MAPPING.get(concept, {}).keys()
        
        for case in cases:
            synsets = SYNSET_MAPPING[concept][case]
            
            # Se all'interno hai un loop che conta quante immagini stai salvando 
            # fino a raggiungere 'target_imgs', puoi mettere una seconda barra:
            
            # pbar_images = tqdm(total=target_imgs, desc=f" -> {case}", leave=False)
            
            # Esempio di loop interno (adatta alla tua logica reale):
            images_collected = 0
            while images_collected < target_imgs:
                # ... logica di estrazione/aumentazione ...
                
                # Quando salvi un'immagine con successo:
                images_collected += 1
                # pbar_images.update(1) # Avanza di 1 sulla barra interna
                
            # Chiudi la barra interna una volta completato il caso
            # pbar_images.close()

In [ ]:
def generate_dual_dataset(pipe: Pipeline, target_imgs: int = 120, mode: str = "vanilla"):
    """
    Main entry point per l'estrazione e l'aumentazione del dataset.
    Itera sulla struttura dati SYNSET_MAPPING (concept -> case -> synsets).

    Modalità supportate:
    - mode="vanilla": Genera crop puri dalle immagini originali (fino a target_imgs).
    - mode="pure_augmentation": Genera target_imgs immagini applicando trasformazioni 
      partendo dai crop "vanilla" generati in precedenza.
    """
    POLICIES = ["sliding_window", "bbox", "center_mask", "largest_bbox"]
    TARGET_CONCEPTS = ["striped", "dotted", "chequered", "wood", "water", "braided", "bubbly", "fibrous", "veined"]
    
    croppers = {p: Cropper(replace(pipe.config.crop, policy=p)) for p in POLICIES}
    augmenter = Augmenter(replace(pipe.config.augmentation, enabled=True))

    pbar_concepts = tqdm(TARGET_CONCEPTS, desc=f"Pipeline ({mode})")
    
    for concept in pbar_concepts:
        pbar_concepts.set_postfix(current_concept=concept)
        
        cases = SYNSET_MAPPING[concept] #
        labels = LABELS_BASE[concept]   #

        pbar_cases = tqdm(cases.items(), desc=f"Pipeline ({mode})")
    
        for case, synsets in pbar_cases: 
            pbar_concepts.set_postfix(current_case=case)

            print(f"\n>>> Mode: {mode} | Concept: {concept} / {case} <<<")

            collector = DatasetCollector(pipe.config.dataset, {concept: synsets})
            image_paths = collector.collect().get(concept, [])

            saved_crops = {p: [] for p in POLICIES}

            _vanilla_exist = all(
                os.path.isdir(os.path.join(pipe.config.crops_path, "vanilla", f"vanilla_{pol}", concept, case))
                and bool(os.listdir(os.path.join(pipe.config.crops_path, "vanilla", f"vanilla_{pol}", concept, case)))
                for pol in POLICIES
            )
            _skip_extraction = (mode == "pure_augmentation" and _vanilla_exist)

            for img_path in ([] if _skip_extraction else image_paths):
                if all(len(saved_crops[p]) >= target_imgs for p in POLICIES):
                    break

                try:
                    image_array, detections = pipe.model.process(img_path, labels, threshold=0.25)
                except Exception:
                    continue

                if not detections:
                    continue

                for pol in POLICIES:
                    if len(saved_crops[pol]) >= target_imgs:
                        continue

                    # Cartella di destinazione per i crop puri
                    vanilla_dir = os.path.join(pipe.config.crops_path, "vanilla", f"vanilla_{pol}", concept, case)
                    os.makedirs(vanilla_dir, exist_ok=True)

                    try:
                        crop_fn, _ = croppers[pol].crop(image_array, detections, vanilla_dir)
                        saved_crops[pol].append(crop_fn)
                    except ValueError:
                        pass

            for pol in POLICIES:
                n_crops = len(saved_crops[pol])

                if mode == "pure_augmentation":
                    # Creiamo un dataset di target_imgs immagini interamente aumentate
                    augm_dir = os.path.join(pipe.config.crops_path, "augmented", f"augmented_{pol}", concept, case)
                    os.makedirs(augm_dir, exist_ok=True)
                    
                    _exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
                    vanilla_dir = os.path.join(pipe.config.crops_path, "vanilla", f"vanilla_{pol}", concept, case)
                    if n_crops == 0 and os.path.isdir(vanilla_dir):
                        disk_seeds = [
                            os.path.join(vanilla_dir, f)
                            for f in os.listdir(vanilla_dir)
                            if f.lower().endswith(_exts)
                        ]
                        print(f"  [{pol}] Nessun crop nella run corrente; uso {len(disk_seeds)} semi da disco.")
                    else:
                        disk_seeds = saved_crops[pol]

                    if disk_seeds:
                        print(f"  [{pol}] Genero {target_imgs} immagini aumentate da {len(disk_seeds)} semi.")
                        cycle = itertools.cycle(disk_seeds)
                        for _ in range(target_imgs):
                            src = next(cycle)
                            img = Image.open(src).convert("RGB")
                            aug_img = augmenter.augment_one(img)
                            aug_fn = os.path.join(augm_dir, f"aug_{uuid.uuid4().hex[:8]}.png")
                            aug_img.save(aug_fn)
                    else:
                        print(f"  [{pol}] Impossibile aumentare: 0 semi trovati (né in run né su disco).")

                else:
                    print(f"  [{pol}] Dataset vanilla completato: {n_crops} immagini.")

In [ ]:
# 1. Inizializza la pipeline con la configurazione
pipe = Pipeline(config)

# 2. Lancia l'esperimento Vanilla (Estrazione dei crop)
print("=========================================")
print(" INIZIO GENERAZIONE DATASET VANILLA")
print("=========================================")
generate_dual_dataset(pipe, target_imgs=120, mode="vanilla")

# 3. Lancia l'esperimento Augmented (Trasformazione dei crop)
# Assicurati che la run vanilla abbia finito e salvato i file su disco
print("\\n=========================================")
print(" INIZIO GENERAZIONE DATASET AUGMENTED")
print("=========================================")
generate_dual_dataset(pipe, target_imgs=120, mode="pure_augmentation")

### 11.5 Visualise any experiment's outputs

In [ ]:
def show_experiment(experiment_path: str, n_rows: int = 3, n_cols: int = 4):
    """Plot a grid of crops from one experiment, per concept and case."""
    # Iteriamo sul nuovo dizionario unificato
    for concept, cases in SYNSET_MAPPING.items():
        for case in cases.keys():
            # Il percorso ora deve includere sia il concept che il case
            crop_dir = os.path.join(config.crops_path, experiment_path, concept, case)
            
            if not (os.path.isdir(crop_dir) and os.listdir(crop_dir)):
                # Commentato per evitare spam nell'output se una classe non ha detezioni
                # print(f"[{experiment_path}/{concept}/{case}] no images.")
                continue
                
            print(f"\n── {experiment_path} / {concept} / {case} ──")
            plot_images(n_rows, n_cols, crop_dir)

# Esempio di utilizzo
show_experiment("vanilla/vanilla_sliding_window")